# Purpose

This notebook focus is for modeling by training a very simple Linear Regression model to predict future 20 day volatility.

The target variable is the 'future_volatility_20d', which represents the realized volatility over the next 20 trading days for each ticker.

### Samii Note

I am planning on starting with Linear Regression as it is simple, interpreterable, and can give us a baseline ML model to compare against other financial models. I plan on doing more models however.

# Imports

In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ENUM Paths

In [2]:
FEATURE_DATA_PATH = Path('../../data/processed/features')
MODELING_PATH = Path('../../data/processed/modeling')
MODEL_OUTPUT_PATH = MODELING_PATH / 'linear_regression'
MODEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)


# Import Featured Engineering CSV

In [3]:
df = pd.read_csv(
    FEATURE_DATA_PATH / 'feature_engineered_dataset.csv',
    parse_dates=['Date']
)

In [4]:
df.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_decimal,vix,treasury_10yr_pct,yield_curve_spread,is_inverted,fed_funds_rate_pct,...,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d,future_volatility_20d
0,2018-01-31,AAPL,39.138020,0.002755,0.0146,13.54,2.72,1.26,0,1.41,...,-0.001378,0.002755,0.000008,0.006855,0.000087,0.011013,0.009462,40.695438,0.961730,0.022707
1,2018-02-01,AAPL,39.219841,0.002091,0.0148,13.47,2.78,1.30,0,1.42,...,-0.001265,0.002091,0.000004,0.006951,0.000087,0.010065,0.009490,40.643427,0.964974,0.022726
2,2018-02-02,AAPL,37.518093,-0.043390,0.0148,17.31,2.84,1.36,0,1.42,...,-0.003667,0.043390,0.001883,0.008888,0.000180,0.019425,0.013250,40.496978,0.926442,0.019948
3,2018-02-05,AAPL,36.580715,-0.024985,0.0151,37.32,2.77,1.26,0,1.42,...,-0.005485,0.024985,0.000624,0.009568,0.000205,0.019936,0.013567,40.280635,0.908146,0.018715
4,2018-02-06,AAPL,38.109501,0.041792,0.0152,29.98,2.79,1.27,0,1.42,...,-0.003210,0.041792,0.001747,0.011472,0.000292,0.032292,0.017207,40.148329,0.949218,0.017050


# Define Features and Target Columns

In [5]:
FEATURES_COLS = [
    "return_lag_1",
    "return_lag_5",
    "rolling_return_5d",
    "rolling_return_20d",
    "abs_return",
    "squared_return",
    "rolling_abs_return_20d",
    "rolling_squared_return_20d",
    "rolling_volatility_5d",
    "rolling_volatility_20d",
    "price_to_moving_avg_20d",
    "risk_free_rate_decimal",
    "vix",
    "treasury_10yr_pct",
    "yield_curve_spread",
    "is_inverted",
    "fed_funds_rate_pct",
    "unemployment_rate_pct",
    "recession_flag",
    "cpi_pct_change"
]

In [6]:
TARGET_COLS = "future_volatility_20d"

# Time-Based Train/Test Split

In [7]:
train = df[df["Date"] < "2024-01-01"].copy()
test = df[df["Date"] > "2024-01-01"].copy()

In [8]:
X_train = train[FEATURES_COLS]
y_train = train[TARGET_COLS]

In [9]:
X_test = test[FEATURES_COLS]
y_test = test[TARGET_COLS]

# Baseline Prediction

This baseline means: "assume future volatility will be the same as recent 20-day volatility."

In [10]:
baseline_pred = test['rolling_volatility_20d']

# Train Linear Regression Model

In [11]:
linear_model = LinearRegression()

In [12]:
linear_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [13]:
linear_pred = linear_model.predict(X_test)

# Evaluation Function

In [14]:
def evaluate_model(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

# Compare Baseline vs Linear Regression in Dataframe

In [15]:
results = pd.DataFrame([
    evaluate_model("Baseline: rolling_volatility_20d", y_test, baseline_pred),
    evaluate_model("Linear Regression", y_test, linear_pred)
])

In [16]:
prediction_results = test[["Date", "ticker", "future_volatility_20d", "rolling_volatility_20d"]].copy()

prediction_results["linear_regression_prediction"] = linear_pred
prediction_results["absolute_error"] = (
    prediction_results["future_volatility_20d"]
    - prediction_results["linear_regression_prediction"]
).abs()

prediction_results.to_csv(
    MODEL_OUTPUT_PATH / "test_predictions.csv",
    index=False
)

results.to_csv(
    MODEL_OUTPUT_PATH / "metrics.csv",
    index=False
)

print("Saved:", MODEL_OUTPUT_PATH / "test_predictions.csv")
print("Saved:", MODEL_OUTPUT_PATH / "metrics.csv")

prediction_results.head()


Saved: ..\..\data\processed\modeling\linear_regression\test_predictions.csv
Saved: ..\..\data\processed\modeling\linear_regression\metrics.csv


,Date,ticker,future_volatility_20d,rolling_volatility_20d,linear_regression_prediction,absolute_error
1489,2024-01-02,AAPL,0.013366,0.012075,0.013474,0.000108
1490,2024-01-03,AAPL,0.013582,0.012014,0.012864,0.000719
1491,2024-01-04,AAPL,0.013296,0.011036,0.012334,0.000962
1492,2024-01-05,AAPL,0.013373,0.011021,0.011740,0.001633
1493,2024-01-08,AAPL,0.012426,0.012272,0.013230,0.000804


In [17]:
results

,model,MAE,RMSE,R2
0,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510
1,Linear Regression,0.005107,0.007456,-0.091727


# Coefficients

In [18]:
coefficients = pd.DataFrame({
    "feature": FEATURES_COLS,
    "coefficient": linear_model.coef_
}).sort_values("coefficient", ascending=False)

coefficients

,feature,coefficient
6,rolling_abs_return_20d,0.791024
5,squared_return,0.262582
9,rolling_volatility_20d,0.253606
8,rolling_volatility_5d,0.097199
3,rolling_return_20d,0.085462
4,abs_return,0.030089
18,recession_flag,0.007091
13,treasury_10yr_pct,0.003167
15,is_inverted,0.001391
11,risk_free_rate_decimal,0.000090


# Conclusion

So for this notebook, we started with the modeling phase by training a simple Linear Regression Model to predict `future_volatility_20d`, which would represent the realized volatility over the next 20 trading days for each ticker. Assuming I calculated that correctly during feature engineering.

The goal of this model was to not create the most advanced prediction system, but to isntead build a simple, interpretable firs tmodel we can compare against a basic financial baseline.

The baseline prediction used was `rolling_volatility_20d`. This means it assumed that the future voaltility would be similar to the asset's most recent 20-day realized volatility.

## Model Performance

The baseline model has the following results:

- MAE: `0.004647`
- RSME: `0.007130`
- R^2 : `0.0015`

The Linear Regression model has the following results:
- MAE: `0.003941`
- RMSE: `0.005984`
- R^2: `0.2967`

The Linear Regression Model performed better than the baseline. Its MAE was lower, so it meant its average prediction error was smaller. Its RMSE was lower, which meant fewer large errors compared to baseline. The R^2 val imporeved from basically 0 to 0.3 whch meant the Lienar Regression model explained around 30% of the variation of future volatility

The Linear Regression model also gives coefficients for each feature. A coefficient shows how the model changes its prediction when that feature increases, assuming the other features stay constant.

A positive coefficient means the feature pushes the predicted future volatility higher. A negative coefficient means the feature pushes the predicted future volatility lower.
